In [1]:
import pandas as pd

# Load all three cleaned datasets
oy = pd.read_csv('../data/olive_young/oliveyoung_skincare_clean.csv')
sephora = pd.read_csv('../data/sephora/sephora_skincare_clean.csv')
ulta = pd.read_csv('../data/ulta/ulta_skincare_clean.csv')

print("=== OLIVE YOUNG COLUMNS ===")
print(oy.columns.tolist())

print("\n=== SEPHORA COLUMNS ===")
print(sephora.columns.tolist())

print("\n=== ULTA COLUMNS ===")
print(ulta.columns.tolist())

=== OLIVE YOUNG COLUMNS ===
['product_id', 'product_name', 'brand', 'top_category', 'sub_category', 'is_skincare', 'price', 'url', 'ingredients_raw', 'ingredient_count', 'ingredients_list']

=== SEPHORA COLUMNS ===
['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count', 'rating', 'reviews', 'size', 'variation_type', 'variation_value', 'variation_desc', 'ingredients', 'price_usd', 'value_price_usd', 'sale_price_usd', 'limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category', 'secondary_category', 'tertiary_category', 'child_count', 'child_max_price', 'child_min_price']

=== ULTA COLUMNS ===
['product_name', 'website', 'country', 'category', 'subcategory', 'title-href', 'price', 'brand', 'ingredients', 'form', 'type', 'color', 'size', 'rating', 'noofratings']


In [2]:
# Standardize Olive Young
oy_clean = pd.DataFrame({
    'product_name':    oy['product_name'],
    'brand':           oy['brand'],
    'category':        oy['sub_category'],
    'price':           oy['price'],
    'rating':          None,
    'ingredients':     oy['ingredients_raw'],
    'ingredient_count': oy['ingredient_count'],
    'url':             oy['url'],
    'store':           'olive_young'
})

# Standardize Sephora
sephora_clean = pd.DataFrame({
    'product_name':    sephora['product_name'],
    'brand':           sephora['brand_name'],
    'category':        sephora['secondary_category'],
    'price':           sephora['price_usd'],
    'rating':          sephora['rating'],
    'ingredients':     sephora['ingredients'],
    'ingredient_count': sephora['ingredients'].dropna().apply(lambda x: len(str(x).split(','))),
    'url':             None,
    'store':           'sephora'
})

# Standardize Ulta
ulta_clean = pd.DataFrame({
    'product_name':    ulta['product_name'],
    'brand':           ulta['brand'],
    'category':        ulta['subcategory'],
    'price':           ulta['price'],
    'rating':          ulta['rating'],
    'ingredients':     ulta['ingredients'],
    'ingredient_count': ulta['ingredients'].dropna().apply(lambda x: len(str(x).split(','))),
    'url':             ulta['title-href'],
    'store':           'ulta'
})

print(f"Olive Young: {len(oy_clean)} products")
print(f"Sephora: {len(sephora_clean)} products")
print(f"Ulta: {len(ulta_clean)} products")

Olive Young: 1573 products
Sephora: 1899 products
Ulta: 1107 products


In [3]:
# Combine all three
master = pd.concat([oy_clean, sephora_clean, ulta_clean], ignore_index=True)

print(f"Total products: {len(master)}")
print(f"Products with ingredients: {master['ingredients'].notna().sum()}")
print(f"Ingredient coverage: {master['ingredients'].notna().sum() / len(master) * 100:.1f}%")

print("\n=== BY STORE ===")
print(master['store'].value_counts())

print("\n=== SAMPLE ROWS ===")
print(master[['product_name', 'brand', 'category', 'store', 'price']].head(10).to_string(index=False))

Total products: 4579
Products with ingredients: 4368
Ingredient coverage: 95.4%

=== BY STORE ===
store
sephora        1899
olive_young    1573
ulta           1107
Name: count, dtype: int64

=== SAMPLE ROWS ===
                                                             product_name        brand              category       store                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [7]:
# Check what the ingredients actually look like in the master df
print("=== SAMPLE INGREDIENTS (first 200 chars each) ===")
oy_sample = master[master['store'] == 'olive_young']['ingredients'].dropna().head(5)
for i, ing in enumerate(oy_sample):
    print(f"\nProduct {i+1}: {str(ing)[:200]}")

=== SAMPLE INGREDIENTS (first 200 chars each) ===

Product 1: Full Ingredients [beplain Mung Bean PH-balanced Cleansing Foam 2.7 fl. oz. (80 ml)] Glycerin; Water; Sodium Cocoyl Glycinate; Sodium Lauroyl Glutamate; 1,2-Hexanediol; Hydroxypropyl Starch Phosphate; 

Product 2: Full Ingredients [manyo Pure Soybean Cleansing Foam 5.41 fl. oz. (160 ml)] Glycerin; Water(Aqua); Stearic Acid; Myristic Acid; Lauric Acid; Potassium Cocoate; Potassium Hydroxide; Potassium Cocoyl Gly

Product 3: Full Ingredients [Abib Jericho Rose Pha Toner Skin Booster 6.76 fl. oz.(200ml)] Water (Aqua), Propanediol, Butylene Glycol,Gluconolactone, Glycerin, Niacinamide, 2,3-Butanediol, Tro-methamine, 1,2-Hex

Product 4: Full Ingredients [acropass Trouble Care Microcone Patch x 24ea] [Trouble care patch] Sodium Hyaluronate, Arginine, Oligopeptide-76, Niacinamide, Water(Aqua), Sodium Hydroxide, Madecassoside, Allantoin

Product 5: Full Ingredients [SKINFOOD Carrot Carotene Calming Water Pad 60ea 8.81 oz.(250g)] WATE